In [5]:
%cd ..

c:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier


In [6]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

In [7]:
PROJECT_DIR = Path.cwd()
DATA_MAP_PATH = PROJECT_DIR / "data" / "data_map.csv"

df = pd.read_csv(DATA_MAP_PATH)

df["processed_path"] = df["processed_path"].apply(lambda p: str(PROJECT_DIR / p))

In [8]:
train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

print(len(train_df), len(val_df), len(test_df))

4172 1044 624


In [9]:
class XrayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        self.label_map = {
            "NORMAL": 0,
            "PNEUMONIA": 1
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["processed_path"]
        img = Image.open(img_path).convert("L")

        if self.transform:
            img = self.transform(img)

        label = self.label_map[row["class"]]

        return img, label

In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [11]:
train_dataset = XrayDataset(train_df, transform=transform)
val_dataset   = XrayDataset(val_df, transform=transform)
test_dataset  = XrayDataset(test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 1, 224, 224])
torch.Size([32])
tensor([1, 1, 1, 1, 0, 0, 1, 0, 0, 1])
